# Bias detection and mitigation in AutoAI

This notebook contains the steps and code to demonstrate support of AutoAI experiments with bias detection/mitigation in watsonx.ai service. It introduces commands for data retrieval, training experiments, persisting pipelines, testing pipelines and scoring.

Some familiarity with Python is helpful. This notebook uses Python 3.12.


## Learning goals

The learning goals of this notebook are:

-  Work with watsonx.ai experiment to train AutoAI models with bias detection and mitigation.
-  Compare trained models quality and fairness.


## Contents

This notebook contains the following parts:

1. [Set up the environment](#Set-up-the-environment)
2. [Optimizer definition](#Optimizer-definition)
3. [Experiment run](#Experiment-run)
4. [Fairness insights](#Fairness-insights)
5. [Cleanup](#Cleanup)
6. [Summary and next steps](#Summary-and-next-steps)

<a id="Set-up-the-environment"></a>
## Set up the environment

Before you use the sample code in this notebook, you must perform the following setup tasks:

-  Contact with your IBM Cloud Pak® for Data administrator and ask them for your account credentials

### Install dependencies
**Note:** `ibm-watsonx-ai` documentation can be found <a href="https://ibm.github.io/watsonx-ai-python-sdk/index.html" target="_blank" rel="noopener no referrer">here</a>.

In [1]:
%pip install -U wget | tail -n 1
%pip install -U autoai-libs | tail -n 1
%pip install -U ibm-watsonx-ai | tail -n 1
%pip install "scikit-learn==1.6.1" | tail -n 1
%pip install -U "lale[fairness]" | tail -n 1
%pip install -U nbformat | tail -n 1
%pip install -U "setuptools>=65.5.1,<82.0.0" | tail -n 1
%pip install -U plotly | tail -n 1

#### Define credentials

Authenticate the watsonx.ai Runtime service on IBM Cloud Pak® for Data. You need to provide the **admin's** `username` and the platform `url`.

In [ ]:
username = "PASTE YOUR USERNAME HERE"
url = "PASTE THE PLATFORM URL HERE"

Use the **admin's** `api_key` to authenticate watsonx.ai Runtime services:

In [ ]:
import getpass

from ibm_watsonx_ai import Credentials

credentials = Credentials(
    username=username,
    api_key=getpass.getpass("Enter your watsonx.ai API key and hit enter: "),
    url=url,
    instance_id="openshift",
    version="5.4",
)

Alternatively you can use the **admin's** `password`:

In [3]:
import getpass

from ibm_watsonx_ai import Credentials

if "credentials" not in locals() or not credentials.api_key:
    credentials = Credentials(
        username=username,
        password=getpass.getpass("Enter your watsonx.ai password and hit enter: "),
        url=url,
        instance_id="openshift",
        version="5.4",
    )

#### Create `APIClient` instance

In [4]:
from ibm_watsonx_ai import APIClient

client = APIClient(credentials)

### Working with spaces

First of all, you need to create a space that will be used for your work. If you do not have space already created, you can use `{PLATFORM_URL}/ml-runtime/spaces?context=icp4data` to create one.

- Click New Deployment Space
- Create an empty space
- Go to space `Settings` tab
- Copy `space_id` and paste it below

**Tip**: You can also use SDK to prepare the space for your work. More information can be found [here](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.4/notebooks/python_sdk/instance-management/Space%20management.ipynb).

**Action**: Assign space ID below


In [ ]:
space_id = "PASTE YOUR SPACE ID HERE"

You can use the `list` method to print all existing spaces.

In [ ]:
client.spaces.list(limit=10)

To be able to interact with all resources available in watsonx.ai, you need to set the **space** which you will be using.

In [6]:
client.set.default_space(space_id)

'SUCCESS'

<a id="Optimizer-definition"></a>
## Optimizer definition

### Training data sets

Define connection information to training data CSV file.

Download training data from git repository.

In [ ]:
import os

import wget

filename = "german_credit_data_biased_training.csv"

url = "https://github.com/IBM/watsonx-ai-samples/raw/master/cpd5.4/data/bias/german_credit_data_biased_training.csv"
if not os.path.isfile(filename):
    wget.download(url)

#### Create data asset

In [8]:
asset_details = client.data_assets.create(
    "german_credit_data_biased_training.csv", filename
)
asset_details

Creating data asset...
SUCCESS


{'metadata': {'space_id': '1ada6f36-e925-480a-92ad-f1d6db6ea290',
  'usage': {'last_updated_at': '2026-04-23T11:27:05Z',
   'last_updater_id': '1000331001',
   'last_update_time': 1776943625570,
   'last_accessed_at': '2026-04-23T11:27:05Z',
   'last_access_time': 1776943625570,
   'last_accessor_id': '1000331001',
   'access_count': 0},
  'rov': {'mode': 0,
   'collaborator_ids': {},
   'member_roles': {'1000331001': {'user_iam_id': '1000331001',
     'roles': ['OWNER']}}},
  'is_linked_with_sub_container': False,
  'name': 'german_credit_data_biased_training.csv',
  'description': '',
  'asset_type': 'data_asset',
  'origin_country': 'us',
  'resource_key': 'german_credit_data_biased_training.csv',
  'rating': 0.0,
  'total_ratings': 0,
  'catalog_id': 'be01af28-96c0-4b78-92db-5c6bef7f05f4',
  'created': 1776943625570,
  'created_at': '2026-04-23T11:27:05Z',
  'owner_id': '1000331001',
  'size': 0,
  'version': 2.0,
  'asset_state': 'available',
  'asset_attributes': ['data_asset'],


In [9]:
client.data_assets.get_id(asset_details)

'c01e0dc4-6d46-410b-9791-2766a885b9cb'

In [10]:
from ibm_watsonx_ai.helpers import DataConnection

german_credit_risk = DataConnection(
    data_asset_id=client.data_assets.get_id(asset_details)
)

training_data_reference = [german_credit_risk]

### Bias detection and mitigation 

#### Terms and definitions:

`Fairness Attribute` - Bias or fairness is typically measured using some fairness attribute such as Gender, Ethnicity, Age, etc. 

`Monitored/Reference Group` - Monitored group are those values of fairness attribute for which we want to measure bias. The rest of the values of the fairness attributes are called as reference group. In case of Fairness Attribute=Gender, if we are trying to measure bias against females, then Monitored group is “Female” and Reference group is “Male”.

`Favourable/Unfavourable outcome` - An important concept in bias detection is that of favourable and unfavourable outcome of the model. E.g., Claim approved can be considered as a favourable outcome and Claim denied can be considered as an unfavourable outcome.

`Disparate Impact` - metric used to measure bias (computed as the ratio of percentage of favourable outcome for the monitored group to the percentage of favourable outcome for the reference group). Bias is said to exist if the disparate impact value is below some threshold.

#### Optimizer configuration

Provide input information for AutoAI optimizer:
- `name` - experiment name
- `prediction_type` - type of the problem
- `prediction_column` - target column name
- `fairness_info` - bias detection configuration
- `scoring` - `accuracy_and_disparate_impact` combined optimization metric for both accuracy and fairness. For  regression learning problem the `r2_and_disparate_impact` metric is supported (combines r2 and fairness).

#### `fairness_info` definition:

 - `protected_attributes` (list of dicts) – subset of features for which fairness calculation is desired.
     - `feature` - name of feature for which `reference_group` and `monitored_group` are specified.
     - `reference_group` and `monitored_group` - monitored group are those values of fairness attribute for which we want to measure bias. The rest of the values of the fairness attribute are reference group.
     
 - `favorable_labels` and `unfavorable_labels` – label values which are considered favorable (i.e. “positive”). `unfavorable_labels` are required when prediction type is regression.  
     
Examples of supported configuration:
```
fairness_info = {
            "protected_attributes": [
                {"feature": "Age", "reference_group": [[26, 26], [30, 75]], 
                                    "monitored_group": [[18, 25], [27, 29]]}
            ],
            "favorable_labels": ["No Risk"]
            }

```

```
fairness_info = {
            "protected_attributes": [
                {"feature": "sex", "reference_group": ['male', 'not specified'], 
                                   "monitored_group": ['female']},
                {"feature": "age", "reference_group": [[26, 100]], "monitored_group": [[18, 25], [27, 29]]}
            ],
            "favorable_labels": [[5000.01, 9000]],
            "unfavorable_labels": [[0, 5000], [9000, 1000000]]
            }

```

In [11]:
fairness_info = {
    "protected_attributes": [
        {"feature": "Sex", "reference_group": ["male"], "monitored_group": ["female"]},
        {
            "feature": "Age",
            "reference_group": [[26, 75]],
            "monitored_group": [[18, 25]],
        },
    ],
    "favorable_labels": ["No Risk"],
    "unfavorable_labels": ["Risk"],
}

In [12]:
from ibm_watsonx_ai.experiment import AutoAI

experiment = AutoAI(credentials, space_id=space_id)

pipeline_optimizer = experiment.optimizer(
    name="Credit Risk Prediction and bias detection - AutoAI",
    prediction_type=AutoAI.PredictionType.BINARY,
    prediction_column="Risk",
    scoring="accuracy_and_disparate_impact",
    fairness_info=fairness_info,
    max_number_of_estimators=1,
    retrain_on_holdout=False,
    include_only_estimators=["XGBClassifier"],
)

<a id="Experiment-run"></a>
## Experiment run

Call the `fit()` method to trigger the AutoAI experiment. You can either use interactive mode (synchronous job) or background mode (asychronous job) by specifying `background_model=True`.

In [13]:
run_details = pipeline_optimizer.fit(
    training_data_reference=training_data_reference, background_mode=False
)

Training job e291e18f-8d53-434f-8bad-b07c1d83661e completed: 100%|████████| [01:56<00:00,  1.16s/it]


You can use the `get_run_status()` method to monitor AutoAI jobs in background mode.

### Get selected pipeline model

Download and reconstruct a scikit-learn pipeline model object from the
AutoAI training job.

In [14]:
experiment_summary = pipeline_optimizer.summary()
experiment_summary.head()

,Enhancements,Estimator,training_accuracy_and_disparate_impact_(optimized),training_disparate_impact_Sex,training_roc_auc,holdout_disparate_impact_Sex,holdout_average_precision,holdout_log_loss,holdout_roc_auc,training_disparate_impact,...,holdout_accuracy,holdout_balanced_accuracy,training_recall,holdout_f1,training_accuracy,holdout_disparate_impact,training_balanced_accuracy,holdout_disparate_impact_Age,training_f1,training_disparate_impact_Age
Pipeline Name,,,,,,,,,,,,,,,,,,,,,
Pipeline_1,,XGBClassifier,0.676887,1.009581,0.846120,1.046499,0.480936,0.419151,0.855620,1.825512,...,0.811623,0.754275,0.894970,0.867606,0.796567,1.431694,0.748223,1.426056,0.853965,2.329145
Pipeline_2,HPO,XGBClassifier,0.676887,1.009581,0.846120,1.046499,0.480936,0.419151,0.855620,1.825512,...,0.811623,0.754275,0.894970,0.867606,0.796567,1.431694,0.748223,1.426056,0.853965,2.329145
Pipeline_3,"HPO, FE",XGBClassifier,0.681126,1.009843,0.846576,1.057825,0.481095,0.416944,0.855187,1.787199,...,0.809619,0.755745,0.892283,0.865248,0.795004,1.463687,0.747200,1.451613,0.852646,2.286728
Pipeline_4,"HPO, FE, HPO",XGBClassifier,0.681126,1.009843,0.846576,1.057825,0.481095,0.416944,0.855187,1.787199,...,0.809619,0.755745,0.892283,0.865248,0.795004,1.463687,0.747200,1.451613,0.852646,2.286728
Pipeline_5,"HPO, FE, HPO, Ensemble",BatchedTreeEnsembleClassifier(XGBClassifier),0.681126,1.009843,0.846576,1.057825,0.481095,0.416944,0.855187,1.787199,...,0.809619,0.755745,0.892283,0.865248,0.795004,1.463687,0.747200,1.451613,0.852646,2.286728


### Visualize pipeline

In [15]:
pipeline_name = experiment_summary.index[
    experiment_summary.holdout_disparate_impact.argmax()
]
best_pipeline = pipeline_optimizer.get_pipeline(pipeline_name=pipeline_name)
best_pipeline.export_to_sklearn_pipeline()

Pipeline(steps=[('featureunion',
                 FeatureUnion(transformer_list=[('float32_transform_4928926880',
                                                 Pipeline(steps=[('numpycolumnselector',
                                                                  NumpyColumnSelector(columns=[0,
                                                                                               1,
                                                                                               2,
                                                                                               3,
                                                                                               5,
                                                                                               6,
                                                                                               7,
                                                                                               8,
                                                                                               9,
                                                                                               10,
                                                                                               11,
                                                                                               12,
                                                                                               13,
                                                                                               14,
                                                                                               15,
                                                                                               16,
                                                                                               17,
                                                                                               18,
                                                                                               19])),
                                                                 ('compressstrings',
                                                                  CompressStrings(compress_type='hash',
                                                                                  dtypes_list=['char_str',
                                                                                               'int_num',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'char_str',
                                                                                               'i...
                               feature_types=None, gamma=None, gpu_id=None,
                               grow_policy=None, importance_type='gain',
                               interaction_constraints=None, learning_rate=None,
                               max_bin=None, max_cat_threshold=None,
                               max_cat_to_onehot=None, max_delta_step=None,
                               max_depth=3, max_leaves=None, min_child_weight=1,
                               missing=nan, monotone_constraints=None,
                               multi_strategy=None, n_estimators=100, n_jobs=4,
                               nthread=None, ...))])

Each node in the visualization is a machine-learning operator
(transformer or estimator). Each edge indicates data flow (transformed
output from one operator becomes input to the next).  The input to the
root nodes is the initial dataset and the output from the sink node
is the final prediction.  When you hover the mouse pointer over a
node, a tooltip shows you the configuration arguments of the
corresponding operator (tuned hyperparameters). When you click on the
hyperlink of a node, it brings you to a documentation page for the
operator.

### Test pipeline model locally

#### Read the data

In [16]:
data_connections = pipeline_optimizer.get_data_connections()

X_train, X_holdout, y_train, y_holdout = data_connections[0].read(
    with_holdout_split=True
)

  Using cached pyarrow-24.0.0-cp312-cp312-macosx_12_0_x86_64.whl.metadata (3.0 kB)
Using cached pyarrow-24.0.0-cp312-cp312-macosx_12_0_x86_64.whl (36.7 MB)


#### Calculate metrics

For detail description of used metrics you can check the documentation:
- [accuracy](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.accuracy_score.html) 
- [disparate_impact](https://lale.readthedocs.io/en/latest/modules/lale.lib.aif360.util.html#lale.lib.aif360.util.disparate_impact)

- [accuracy and disparate impact](https://lale.readthedocs.io/en/latest/modules/lale.lib.aif360.util.html#lale.lib.aif360.util.accuracy_and_disparate_impact)

In [17]:
from lale.lib.aif360 import accuracy_and_disparate_impact, disparate_impact
from sklearn.metrics import accuracy_score

predicted_y = best_pipeline.predict(X_holdout.values)
disparate_impact_scorer = disparate_impact(**fairness_info)
accuracy_disparate_impact_scorer = accuracy_and_disparate_impact(**fairness_info)

print("Accuracy: {:.2f}".format(accuracy_score(y_true=y_holdout, y_pred=predicted_y)))
print(
    "Disparate impact: {:.2f}".format(
        disparate_impact_scorer(best_pipeline, X_holdout, y_holdout)
    )
)
print(
    "Accuracy and disparate impact: {:.2f}".format(
        accuracy_disparate_impact_scorer(best_pipeline, X_holdout, y_holdout)
    )
)

Accuracy: 0.81
Disparate impact: 1.46
Accuracy and disparate impact: 0.75


---

<a id="Fairness-insights"></a>
## Fairness insights

You can analize favorable outcome distributions using `visualize` method from `utils` module.

In [18]:
from ibm_watsonx_ai.utils.autoai.fairness import visualize

visualize(run_details, pipeline_name)

---

<a id="Cleanup"></a>
## Cleanup

If you want to clean up all created assets:
- experiments
- trainings
- pipelines
- model definitions
- models
- functions
- deployments

please follow up this sample [notebook](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.4/notebooks/python_sdk/instance-management/Machine%20Learning%20artifacts%20management.ipynb).

<a id="Summary-and-next-steps"></a>
## Summary and next steps

You successfully completed this notebook!
 
As a next step you can deploy and score the model: [Sample notebook](https://github.com/IBM/watsonx-ai-samples/blob/master/cpd5.4/notebooks/python_sdk/experiments/autoai/Use%20AutoAI%20and%20Lale%20to%20predict%20credit%20risk.ipynb).

Check out our _<a href="https://ibm.github.io/watsonx-ai-python-sdk/samples.html" target="_blank" rel="noopener no referrer">Online Documentation</a>_ for more samples, tutorials, documentation, how-tos, and blog posts.

### Authors
**Lukasz Cmielowski, PhD**, is an Automation Architect and Data Scientist at IBM with a track record of developing enterprise-level applications that substantially increases clients' ability to turn data into actionable knowledge.

**Dorota Lączak**, Software Engineer at watsonx.ai.

**Szymon Kucharczyk**, Software Engineer at watsonx.ai.

**Rafał Chrzanowski**, Software Engineer at watsonx.ai.

Copyright © 2021-2026 IBM. This notebook and its source code are released under the terms of the MIT License.